# Credit Card Fraud Analysis

This notebook explores transaction patterns associated with credit card fraud using pandas. The analysis focuses on transaction volume, fraud incidence, transaction amount, payment channel, merchant category, and combinations of risk signals.

The goal is descriptive analysis and business interpretation. No predictive model is developed in this project.


## 1. Analysis Objectives

- Validate the dataset structure and data quality.
- Measure the frequency and rate of fraudulent transactions.
- Compare transaction amounts for fraudulent and non-fraudulent activity.
- Identify higher-risk channels and merchant categories.
- Examine how fraud rates change when multiple risk signals occur together.


## 2. Setup and Data Loading


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/credit_card_fraud_2026.csv")

In [3]:
df.head() #查看数据集前几行数据

,transaction_id,amount_usd,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,hours_since_last_txn,txn_count_last_24h,...,ip_country_mismatch,billing_shipping_mismatch,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,is_ai_generated_scam_attempt,merchant_risk_score,prior_disputes,is_fraud
0,1,42.86,Restaurants,Visa,OTP,Online,Android Phone,False,13.54,2,...,False,False,0,0.1,18,3,False,42.3,0,0
1,2,4.75,Online Retail,Mastercard,3D Secure,Online,Android Phone,False,0.71,2,...,False,False,0,25.8,12,4,False,28.3,0,0
2,3,77.18,Groceries,Mastercard,3D Secure,Online,Mac,False,0.35,5,...,False,True,0,42.3,5,0,False,24.7,1,0
3,4,1.69,Streaming,Visa,No Authentication,POS,Android Phone,False,3.42,6,...,False,False,0,28.9,22,6,False,56.2,1,0
4,5,261.68,Travel,Visa,3D Secure,In-App,iPhone,False,2.43,2,...,False,False,0,3.9,2,4,False,32.7,0,0


## 3. Data Quality Assessment


In [4]:
df.shape #返回数据的行数和列数

(20000, 26)

In [5]:
df.info() #打印数据集的数据类型、列名、整体内存占用大小

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   transaction_id                20000 non-null  int64  
 1   amount_usd                    20000 non-null  float64
 2   merchant_category             20000 non-null  object 
 3   card_type                     20000 non-null  object 
 4   auth_method                   20000 non-null  object 
 5   channel                       20000 non-null  object 
 6   device_type                   20000 non-null  object 
 7   is_foreign_transaction        20000 non-null  bool   
 8   hours_since_last_txn          20000 non-null  float64
 9   txn_count_last_24h            20000 non-null  int64  
 10  distance_from_home_km         20000 non-null  float64
 11  card_age_months               20000 non-null  int64  
 12  customer_age                  20000 non-null  int64  
 13  a

In [6]:
df.isnull().sum() #统计缺失值数量

transaction_id                  0
amount_usd                      0
merchant_category               0
card_type                       0
auth_method                     0
channel                         0
device_type                     0
is_foreign_transaction          0
hours_since_last_txn            0
txn_count_last_24h              0
distance_from_home_km           0
card_age_months                 0
customer_age                    0
account_balance_usd             0
is_new_merchant                 0
used_vpn                        0
ip_country_mismatch             0
billing_shipping_mismatch       0
cvv_retry_count                 0
velocity_score                  0
time_of_day_hour                0
day_of_week                     0
is_ai_generated_scam_attempt    0
merchant_risk_score             0
prior_disputes                  0
is_fraud                        0
dtype: int64

In [7]:
df.duplicated().sum() #重复数据

np.int64(0)

### Data quality summary

The dataset contains 20,000 rows and 26 columns. The checks above identify no missing values and no duplicate rows, so no imputation or duplicate removal is required before the descriptive analysis.


## 4. Transaction and Fraud Overview


In [8]:
df["is_fraud"].value_counts() #目标变量分布

is_fraud
0    19661
1      339
Name: count, dtype: int64

In [9]:
df["is_fraud"].value_counts(normalize=True) #欺诈交易占比

is_fraud
0    0.98305
1    0.01695
Name: proportion, dtype: float64

Fraudulent transactions account for a small share of the dataset: 339 of 20,000 transactions, or 1.695%. This imbalance is important when interpreting segment-level fraud rates.


## 5. Transaction Amount Analysis


In [10]:
df["amount_usd"].describe()  #整体交易金额的分布：最小值、最大值、均值、中位数

count    20000.000000
mean       132.424597
std        256.963666
min          1.000000
25%         26.235000
50%         57.510000
75%        131.752500
max       6872.690000
Name: amount_usd, dtype: float64

In [11]:
df.groupby("is_fraud")["amount_usd"].mean()   #正常交易和欺诈交易的平均金额有没有明显差异

is_fraud
0    131.582739
1    181.249853
Name: amount_usd, dtype: float64

In [12]:
df.groupby("is_fraud")["amount_usd"].median()  #中位数

is_fraud
0    57.35
1    69.77
Name: amount_usd, dtype: float64

In this dataset, fraudulent transactions have both a higher mean amount and a higher median amount than non-fraudulent transactions. Transaction amount may therefore be useful as one contextual risk factor, but it should not be used as a stand-alone rule.


## 6. Fraud Rate by Channel


In [13]:
df.groupby("channel")["is_fraud"].agg(["count", "sum", "mean"])

,count,sum,mean
channel,,,
ATM,1374,25,0.018195
Contactless,3400,48,0.014118
In-App,3225,52,0.016124
Online,6810,135,0.019824
POS,5191,79,0.015219


Online transactions have the highest fraud rate among the five channels and also represent the largest transaction volume. ATM transactions have the second-highest fraud rate. These results make online activity a priority area for monitoring in this sample.


## 7. Fraud Rate by Merchant Category


In [14]:
merchant_fraud = (
    df.groupby("merchant_category")["is_fraud"]
      .agg(["count", "sum", "mean"])
      .reset_index()
      .rename(columns={
          "count": "transaction_count",
          "sum": "fraud_count",
          "mean": "fraud_rate"
      })
      .sort_values("fraud_rate", ascending=False, ignore_index=True)
)

merchant_fraud

,merchant_category,transaction_count,fraud_count,fraud_rate
0,Crypto Exchange,561,28,0.049911
1,Gift Cards,617,27,0.043760
2,Gaming,1036,38,0.036680
3,Electronics,2032,48,0.023622
4,Streaming,1010,21,0.020792
5,Utilities,1330,25,0.018797
6,Healthcare,1237,16,0.012935
7,Groceries,3242,40,0.012338
8,Fuel,1565,19,0.012141
9,Restaurants,2534,29,0.011444


Crypto Exchange has the highest observed fraud rate, followed by Gift Cards and Gaming. Crypto Exchange contains more than 500 transactions in the dataset, but category-level results should still be interpreted together with transaction counts rather than by rate alone.


## 8. Fraud Rate by Risk Signals


### 8.1 Foreign transactions


In [15]:
df.groupby("is_foreign_transaction")["is_fraud"].agg(["count", "sum", "mean"]) #境外交易和境内交易的欺诈率差异

,count,sum,mean
is_foreign_transaction,,,
False,18760,266,0.014179
True,1240,73,0.058871


Foreign transactions show a higher fraud rate than domestic transactions in this dataset, indicating that cross-border activity is a relevant monitoring signal.


### 8.2 VPN usage


In [16]:
df.groupby("used_vpn")["is_fraud"].agg(["count", "sum", "mean"])  #交易使用VPN

,count,sum,mean
used_vpn,,,
False,18241,262,0.014363
True,1759,77,0.043775


Transactions using a VPN have a higher observed fraud rate than transactions without VPN usage.


### 8.3 IP country mismatch


In [17]:
df.groupby("ip_country_mismatch")["is_fraud"].agg(["count", "sum", "mean"]) #IP所在国家与持卡人/交易预期国家不一致

,count,sum,mean
ip_country_mismatch,,,
False,18834,248,0.013168
True,1166,91,0.078045


Transactions with an IP country mismatch have a higher observed fraud rate than transactions without a mismatch.


### 8.4 Combined risk signals


In [18]:
df.groupby(
    ["used_vpn", "ip_country_mismatch", "is_foreign_transaction"]
)["is_fraud"].agg(["count", "sum", "mean"])

count  sum      mean
used_vpn ip_country_mismatch is_foreign_transaction                      
False    False               False                   16398  168  0.010245
                             True                      782   19  0.024297
         True                False                     710   36  0.050704
                             True                      351   39  0.111111
True     False               False                    1576   51  0.032360
                             True                       78   10  0.128205
         True                False                      76   11  0.144737
                             True                       29    5  0.172414

Fraud rates rise when VPN usage, IP country mismatch, and foreign-transaction signals overlap. The combination containing all three signals has the highest observed rate, but it contains only 29 transactions. High-risk combinations should therefore be interpreted with their sample sizes and used for multi-factor review rather than automatic blocking based on a single flag.


## 9. Key Findings

- Fraud represents 1.695% of all transactions in the dataset.
- Fraudulent transactions have higher mean and median amounts than non-fraudulent transactions.
- Online is the highest-risk channel by observed fraud rate and also has the largest transaction volume.
- Crypto Exchange, Gift Cards, and Gaming have the three highest observed merchant-category fraud rates.
- Foreign transactions, VPN usage, and IP country mismatch are each associated with higher observed fraud rates.
- Multiple overlapping risk signals correspond to higher fraud rates, although the smallest segments require cautious interpretation.
